In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import sys
if ".." not in sys.path:
    sys.path.append("..")
import numpy as np, random, tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from src.evaluacion.metricas import evaluar
import xgboost as xgb
from src.features.feature_sets import get_feature_sets

In [ ]:
df =pd.read_parquet("../data/processed/tabla_features.parquet").sort_values("datetime_utc")
df = df[df["entrenable"]]

In [ ]:
columnas = df.columns.tolist()
# columnas = [col for col in columnas if col.endswith("prevista")]
columnas

In [ ]:
demanda = df[["datetime_utc","demanda_prevista","demanda_prevista_diaria"]]
eolica = df[["datetime_utc","eolica_prevista","eolica_prevista_d1"]]
print(demanda.head())
print(eolica.head())

In [ ]:
RAW = Path("../data/raw/esios")

indicadores = [
    "demanda_prevista", "demanda_prevista_diaria",
    "eolica_prevista",  "eolica_prevista_d1",
    "solar_fv_prevista", "solar_termica_prevista",
]

def plano(v):
    """ESIOS anida 'tiempo'/'magnitud' como dict; devuelve algo legible."""
    if isinstance(v, dict):
        return v.get("name") or v.get("id") or str(v)
    return v


meta, series = [], {}
for nombre in indicadores:
    d = json.loads((RAW / f"esios_datos_{nombre}_raw.json").read_text(encoding="utf-8"))
    ind = d.get("indicator", d)
    meta.append({
        "indicador":   nombre,
        "id":          ind.get("id"),
        "tiempo":      plano(ind.get("tiempo")),
        "step_type":   plano(ind.get("step_type")),
        "magnitud":    plano(ind.get("magnitud")),
        "n_valores":   len(ind["values"]),
    })
    s = (pd.DataFrame(ind["values"])[["datetime_utc", "value"]]
           .assign(datetime_utc=lambda x: pd.to_datetime(x["datetime_utc"], utc=True))
           .set_index("datetime_utc")["value"]
           .rename(nombre))
    series[nombre] = s

print("METADATOS ESIOS ------------------------------------------------------")
print(pd.DataFrame(meta).to_string(index=False))


def comparar(a, b):
    par = pd.concat([series[a], series[b]], axis=1).dropna()
    x, y = par[a], par[b]
    ratio = (x / y).replace([np.inf, -np.inf], np.nan).dropna()
    print(f"\n{a}  vs  {b}")
    print(f"  n solapado       : {len(par)}")
    print(f"  correlacion r    : {x.corr(y):.5f}")
    print(f"  ratio a/b medio  : {ratio.mean():.4f}  (std {ratio.std():.4f})")
    print(f"  medias crudas    : {a}={x.mean():.0f} | {b}={y.mean():.0f}")

print("\nCOMPARACION DE VERSIONES --------------------------------------------")
comparar("demanda_prevista", "demanda_prevista_diaria")
comparar("eolica_prevista",  "eolica_prevista_d1")


In [ ]:
# --- Diagnóstico de duplicados en datetime_utc (ESIOS) -----------------------
for nombre, s in series.items():
    dup = s.index.duplicated(keep=False)
    n_dup = dup.sum()
    print(f"{nombre:25s}: {n_dup} filas con datetime_utc duplicado")
    if n_dup:
        # ¿los duplicados tienen el MISMO valor o distinto (vintage)?
        muestra = s[dup].groupby(level=0).nunique()
        con_valores_distintos = (muestra > 1).sum()
        print(f"    -> de esos timestamps, {con_valores_distintos} tienen VALORES DISTINTOS")
        print(s[dup].head(6).to_string())

In [ ]:
def dedup(s):
    """Colapsa datetime_utc duplicados quedándose con el primero."""
    return s[~s.index.duplicated(keep="first")].sort_index()

def comparar(a, b):
    x, y = dedup(series[a]), dedup(series[b])
    par = pd.concat([x, y], axis=1, join="inner").dropna()
    xa, yb = par[a], par[b]
    ratio = (xa / yb).replace([np.inf, -np.inf], np.nan).dropna()
    print(f"\n{a}  vs  {b}")
    print(f"  n solapado       : {len(par)}")
    print(f"  correlacion r    : {xa.corr(yb):.5f}")
    print(f"  ratio a/b medio  : {ratio.mean():.4f}  (std {ratio.std():.4f})")
    print(f"  medias crudas    : {a}={xa.mean():.0f} | {b}={yb.mean():.0f}")

comparar("demanda_prevista", "demanda_prevista_diaria")
comparar("eolica_prevista",  "eolica_prevista_d1")

In [ ]:


df = pd.read_parquet("../data/processed/tabla_features.parquet")
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
df = df.sort_values("datetime_utc").set_index("datetime_utc")

TARGET = "precio_espana"                      
EXOGENAS = [
    "demanda_prevista_diaria",                 
    "eolica_prevista_d1",                      
    "solar_fv_prevista",                       
    "solar_termica_prevista",                  
]
CALENDARIO = ["festivo", "hora_sin", "hora_cos",
              "semana_sin", "semana_cos", "mes_sin", "mes_cos"]

FEATURES = [TARGET] + EXOGENAS + CALENDARIO    

# --- Chequeos de sanidad -----------------------------------------------------
faltan = [c for c in FEATURES if c not in df.columns]
print("Columnas ausentes :", faltan if faltan else "ninguna")

datos = df[FEATURES].copy()
print("\nShape             :", datos.shape)
print("\nNaN por columna:")
print(datos.isna().sum()[lambda s: s > 0].sort_values(ascending=False)
      if datos.isna().any().any() else "  (sin NaN)")

# Continuidad horaria: imprescindible para el ventaneo (huecos = ventanas rotas)
paso = datos.index.to_series().diff().value_counts()
print("\nSaltos temporales (deberían ser casi todos 1h):")
print(paso.head())
print("\nRango:", datos.index.min(), "->", datos.index.max())
print("Índice único:", datos.index.is_unique)
# ============================================================================

In [ ]:

cols_nan = datos.columns[datos.isna().any()].tolist()
print("Columnas con NaN antes :", cols_nan)

datos[cols_nan] = datos[cols_nan].interpolate(method="time", limit_area="inside")

print("NaN totales después     :", int(datos.isna().sum().sum()))
print("NaN por columna después :")
print(datos.isna().sum()[lambda s: s > 0] if datos.isna().any().any() else "  (todo a cero)")
# ============================================================================

In [ ]:
for c in cols_nan:
    idx = datos.index[datos[c].isna()]
    print(f"{c}: n={len(idx)} | primer NaN={idx.min()} | último NaN={idx.max()}")
    print(f"   ¿tocan el final de la serie?: {idx.max() == datos.index.max()}")
    print(f"   ¿son consecutivos al final?: {(datos.index[-len(idx):] == idx).all()}")
    # === Descartar las 23h finales sin previsión D+1 ===========================
antes = len(datos)
datos = datos.dropna(subset=cols_nan)     # solo afecta a la cola (filas consecutivas al final)
print(f"Filas: {antes} -> {len(datos)}  (descartadas {antes - len(datos)})")
print("NaN totales:", int(datos.isna().sum().sum()))
print("Nuevo rango:", datos.index.min(), "->", datos.index.max())
print("Índice sigue único y ordenado:", datos.index.is_unique and datos.index.is_monotonic_increasing)
# ============================================================================

In [ ]:


L = 168  # lookback: 1 semana

# 1) Pre-desfase del precio: el canal autorregresivo = precio de t-1 (nunca t)
M = datos[FEATURES].copy()
M["precio_espana"] = M["precio_espana"].shift(24)   

y_full = datos["precio_espana"].to_numpy()          # target real en t
valores = M.to_numpy()                              # (N, 12); precio ya desfasado
idx = datos.index
N = len(datos)

# 2) Deslizar ventanas [ini..i], predecir precio en i
span_ok = pd.Timedelta(hours=L - 1)
X_list, y_list, ts_list = [], [], []
descartadas_hueco = 0

for i in range(L - 1, N):
    ini = i - L + 1
    # 3) Contigüidad: las L horas deben ser consecutivas (si hay hueco, span > L-1h)
    if idx[i] - idx[ini] != span_ok:
        descartadas_hueco += 1
        continue
    ventana = valores[ini:i + 1]          # (L, 12)
    if np.isnan(ventana).any():           # descarta el shift-NaN de la fila 0
        continue
    X_list.append(ventana)
    y_list.append(y_full[i])
    ts_list.append(idx[i])

X = np.asarray(X_list)                    # float64 -> (n, 168, 12)
y = np.asarray(y_list)                    # float64 -> (n,)
ts = pd.DatetimeIndex(ts_list)            # timestamp de cada y (para folds y comparación)

print("X:", X.shape, "| y:", y.shape)
print("Ventanas descartadas por hueco:", descartadas_hueco)
print("Rango de y:", ts.min(), "->", ts.max())

# 4) Chequeo anti-leakage: la ventana termina en t-1, el target es t
k = 1000
t = ts[k]
assert X[k, -1, 0] == datos["precio_espana"].loc[t - pd.Timedelta(hours=24)]
assert y[k] == datos["precio_espana"].loc[t]
print("Anti-leakage OK: canal precio llega a t-1, target es t")
# ============================================================================

In [ ]:
# === Fase 5.2 · GATE (1/2): split holdout + escalado por train ==============


CUTOFF = pd.Timestamp("2026-01-01", tz="UTC")   # últimos ~6 meses como test
# (guarda este CUTOFF: lo reutilizamos tal cual para el MAE de XGBoost)

train_mask = ts < CUTOFF
test_mask  = ts >= CUTOFF

X_tr, y_tr = X[train_mask], y[train_mask]
X_te, y_te = X[test_mask],  y[test_mask]
print("Train:", X_tr.shape, "| Test:", X_te.shape)
print("Test va de", ts[test_mask].min(), "a", ts[test_mask].max())

# --- Escalado: se AJUSTA solo con train (fit), se aplica a ambos -------------
n_tr, Lw, F = X_tr.shape
sx = StandardScaler().fit(X_tr.reshape(-1, F))          # (n*168, 12)
X_tr_s = sx.transform(X_tr.reshape(-1, F)).reshape(X_tr.shape)
X_te_s = sx.transform(X_te.reshape(-1, F)).reshape(X_te.shape)

sy = StandardScaler().fit(y_tr.reshape(-1, 1))          # target aparte, para desescalar
y_tr_s = sy.transform(y_tr.reshape(-1, 1)).ravel()

print("Escalado OK | media~0, std~1 en train:",
      round(X_tr_s.mean(), 4), round(X_tr_s.std(), 4))
# ============================================================================

In [ ]:
# === Fase 5.2 · GATE (2/2): LSTM simple, entrenar y medir ===================


SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

model = Sequential([
    Input(shape=(Lw, F)),      # (168, 12)
    LSTM(32),                  # empezamos SIMPLE: una capa, 32 unidades
    Dense(1),
])
model.compile(optimizer="adam", loss="mae")   # optimizamos la métrica que comparamos
model.summary()

es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
hist = model.fit(
    X_tr_s, y_tr_s,
    validation_split=0.1,      # Keras usa el 10% MÁS RECIENTE del train como val (temporal, sin barajar antes)
    epochs=50, batch_size=64,
    callbacks=[es], verbose=1,
)

# Predecir y DESescalar antes de medir (el modelo trabaja en escala estandarizada)
pred_s = model.predict(X_te_s).ravel()
pred = sy.inverse_transform(pred_s.reshape(-1, 1)).ravel()

print("\n=== LSTM holdout ===")
print(evaluar(y_te, pred))
# ============================================================================

In [ ]:
# === Veredicto del gate: XGBoost vs LSTM en el MISMO holdout ================


TARGET = "precio_espana"

dfx = pd.read_parquet("../data/processed/tabla_features.parquet")
dfx["datetime_utc"] = pd.to_datetime(dfx["datetime_utc"], utc=True)
dfx = dfx.sort_values("datetime_utc").set_index("datetime_utc")

feats = get_feature_sets(dfx)["predictivo"]          # el set de 102 (previstas D+1, lags, calendario)
ent = dfx[dfx["entrenable"]]                          # mismas filas entrenables que en Fase 5.1

tr = ent[ent.index <  CUTOFF]
te = ent[ent.index >= CUTOFF]

modelo_xgb = xgb.XGBRegressor(                        # <-- pon aquí TUS hiperparámetros del nb 06
    n_estimators=600, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
modelo_xgb.fit(tr[feats], tr[TARGET])
pred_xgb = pd.Series(modelo_xgb.predict(te[feats]), index=te.index)

# --- Alinear ambos modelos sobre las MISMAS horas de test -------------------
pred_lstm = pd.Series(pred, index=ts[test_mask])     # pred y ts ya los tienes del gate
comun = pred_lstm.index.intersection(pred_xgb.index)

y_comun = dfx.loc[comun, TARGET]
print("Horas comunes de test:", len(comun))
print("XGBoost holdout :", evaluar(y_comun, pred_xgb.loc[comun]))
print("LSTM    holdout :", evaluar(y_comun, pred_lstm.loc[comun]))
# ============================================================================

In [ ]:
# === Robustez a la semilla: LSTM en el holdout con varias semillas ==========
def construir_modelo():
    m = Sequential([Input(shape=(Lw, F)), LSTM(32), Dense(1)])
    m.compile(optimizer="adam", loss="mae")
    return m

SEEDS = [42, 0, 1, 2]
resultados = []

for s in SEEDS:
    np.random.seed(s); random.seed(s); tf.random.set_seed(s)
    modelo = construir_modelo()
    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    modelo.fit(X_tr_s, y_tr_s, validation_split=0.1, epochs=50, batch_size=64,
               callbacks=[es], verbose=0)                      # verbose=0: sin ruido por época
    p = sy.inverse_transform(modelo.predict(X_te_s, verbose=0).reshape(-1, 1)).ravel()
    p_lstm = pd.Series(p, index=ts[test_mask]).loc[comun]
    met = evaluar(y_comun, p_lstm)
    resultados.append({"seed": s, "MAE": met["MAE"], "RMSE": met["RMSE"], "sMAPE": met["sMAPE"]})
    print(f"seed {s}: MAE={met['MAE']:.3f}  RMSE={met['RMSE']:.3f}")

res = pd.DataFrame(resultados)
mae_xgb = evaluar(y_comun, pred_xgb.loc[comun])["MAE"]
print("\n--- LSTM sobre semillas ---")
print(f"MAE media {res['MAE'].mean():.3f} ± {res['MAE'].std():.3f}  (min {res['MAE'].min():.3f}, max {res['MAE'].max():.3f})")
print(f"XGBoost MAE (referencia fija): {mae_xgb:.3f}")
# ============================================================================

In [ ]:
# === Ensemble de semillas: ¿promediar mata la varianza y recupera ventaja? ==
preds_seeds = []
for s in SEEDS:
    np.random.seed(s); random.seed(s); tf.random.set_seed(s)
    modelo = construir_modelo()
    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    modelo.fit(X_tr_s, y_tr_s, validation_split=0.1, epochs=50, batch_size=64,
               callbacks=[es], verbose=0)
    p = sy.inverse_transform(modelo.predict(X_te_s, verbose=0).reshape(-1, 1)).ravel()
    preds_seeds.append(pd.Series(p, index=ts[test_mask]).loc[comun])

ensemble = pd.concat(preds_seeds, axis=1).mean(axis=1)   # media de las 4 predicciones
print("Ensemble LSTM :", evaluar(y_comun, ensemble))
print("XGBoost       :", evaluar(y_comun, pred_xgb.loc[comun]))
# ============================================================================

In [ ]:
# Guardar X, y, ts para subir a Colab (X en float32 para que pese la mitad)
np.savez_compressed(
    "../reports/lstm_tensores.npz",
    X=X.astype("float32"),
    y=y.astype("float32"),
    ts=ts.asi8,                 # int64 nanosegundos -> se reconstruye fácil en Colab
)
print("Guardado en reports/lstm_tensores.npz")